# ComfyUI Colab Studio

Self-contained. Nothing to upload. **Runtime > Run all**, then use the
generate cell at the bottom. No public URL is opened unless you turn one
on in cell 1.

This notebook does **images** only. Video generation is not yet shipped in
Colab Studio.

| Cell | What it does |
|---|---|
| 1 | Pick model, persistence, ComfyUI pin, and public-tunnel options |
| 2 | Detect GPU and disk |
| 3 | Install ComfyUI, pinned to the tested revision by default |
| 4 | Link Drive (optional persistence) |
| *(6 unnumbered)* | `%%writefile` the helper library into `colab_studio/` |
| 5 | Choose profile, re-check disk, download models |
| 6 | Write API workflows for the generate cells |
| 7 | Start the server **in the background**; open a public tunnel only if `OPEN_PUBLIC_UI` is on |
| 8 | Generate an image without leaving this notebook |
| 8b | Image-to-image / ControlNet - **tick `run_this` first**; Run all skips it |
| 9-10 | Logs, restart, free VRAM, disk usage, re-tunnel |
| *(last)* | Handbook: model sizes, settings, error fixes |

In [ ]:
#@title 1. Config
IMAGE_MODEL = "auto"  #@param ["auto", "sdxl", "flux-dev", "flux-schnell"]
PERSIST = "outputs-only"  #@param ["outputs-only", "everything", "off"]
CONTROLNET = False  #@param {type:"boolean"}
USE_UPSCALER = True  #@param {type:"boolean"}
COMFY_REF = "pinned"  #@param ["pinned", "latest"]
OPEN_PUBLIC_UI = False  #@param {type:"boolean"}
PORT = 8188
COMFY_DIR = "/content/ComfyUI"
print(f"model={IMAGE_MODEL} persist={PERSIST} "
      f"controlnet={CONTROLNET} upscaler={USE_UPSCALER}")
print(f"comfy_ref={COMFY_REF} open_public_ui={OPEN_PUBLIC_UI}")
if OPEN_PUBLIC_UI:
    print("!! OPEN_PUBLIC_UI is on: cell 7 will start a public Cloudflare "
          "tunnel. The URL is not authentication -- see cell 7's warning.")

In [ ]:
#@title 2. Preflight - GPU, disk, and what actually fits
import shutil, torch

if not torch.cuda.is_available():
    print("!! No GPU. Runtime > Change runtime type > GPU, then rerun.")
    VRAM_GB, GPU_NAME = 0.0, "cpu"
else:
    props = torch.cuda.get_device_properties(0)
    GPU_NAME, VRAM_GB = props.name, props.total_memory / 2**30

# Indicative only. Cell 5 re-measures at the real models/ destination, which
# may be a Drive symlink by then.
DISK_FREE_GB = shutil.disk_usage("/content").free / 2**30
print(f"GPU:  {GPU_NAME}  ({VRAM_GB:.1f} GB VRAM)")
print(f"Disk: {DISK_FREE_GB:.0f} GB free on /content")

In [ ]:
#@title 3. Install ComfyUI
import os
%cd /content
if not os.path.isdir(COMFY_DIR):
    !git clone https://github.com/comfyanonymous/ComfyUI.git {COMFY_DIR}
%cd {COMFY_DIR}

# Kept in sync with colab_studio/compat.py -- that module is the source of
# truth; this literal is generated from it, not hand-typed twice.
TESTED_REF = "806e092ed42772e4ce7abf44c97c50021cc4bd10"
if COMFY_REF == "pinned":
    !git fetch --quiet origin
    !git checkout --quiet {TESTED_REF}
else:
    print("!" * 70)
    print("!! UNSUPPORTED: COMFY_REF='latest'. Colab Studio's workflow graphs")
    print("!! were structurally validated only against 806e092ed427 "
          "(2026-07-26).")
    print("!! Upstream node contracts, input names, or defaults may have")
    print("!! changed since then -- generation can fail in ways nobody here")
    print("!! has tested. Set COMFY_REF='pinned' in cell 1 unless you")
    print("!! specifically need something newer than that.")
    print("!" * 70)
    !git fetch --quiet origin master
    !git checkout --quiet FETCH_HEAD

resolved = !git rev-parse HEAD
print("ComfyUI commit:", resolved[0])

!pip install -q -r requirements.txt
!pip install -q huggingface_hub torchsde requests
# Core ComfyUI only: every node these workflows use ships with it, so there
# are no custom_nodes clones to wait on.
os.makedirs("colab_studio", exist_ok=True)
open("colab_studio/__init__.py", "a").close()
print("ComfyUI installed at", COMFY_DIR)

In [ ]:
#@title 4. Persistence (Drive)
# Models stay on VM disk by default: one Flux checkpoint is 16 GB and the
# free Drive tier is 15 GB, so symlinking models/ fills the quota instantly
# and streams every read over FUSE.
import os

def _link(src, dest):
    os.makedirs(src, exist_ok=True)
    if os.path.isdir(dest) and not os.path.islink(dest):
        os.system(f'cp -rn "{dest}"/* "{src}"/ 2>/dev/null')
        os.system(f'rm -rf "{dest}"')
    if not os.path.islink(dest):
        os.symlink(src, dest)

if PERSIST != "off":
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = "/content/drive/MyDrive/ComfyUI"
    subs = ["output", "user"]
    if PERSIST == "everything":
        print("!! Models on Drive: slow (FUSE) and Flux alone is 16 GB.")
        subs += ["models"]
    for sub in subs:
        _link(os.path.join(DRIVE, sub), os.path.join(COMFY_DIR, sub))
    print("Persisting:", ", ".join(subs))
else:
    print("No persistence - everything is lost when the runtime recycles.")

In [ ]:
%%writefile colab_studio/compat.py
"""Compatibility record: the exact upstream ComfyUI revision this fork's
workflow graphs, node input names, and launch flags were validated against.

ComfyUI's node registry, `INPUT_TYPES()` keys, and default values are not a
stable API -- upstream renames inputs, changes defaults, and removes nodes
without a deprecation cycle. `colab_studio/workflows.py`'s graphs were
structurally validated (`execution.validate_prompt`, the exact code path
`POST /prompt` uses) against exactly `TESTED_REF`. Running against a
different revision means running against node contracts nobody here has
checked -- pinning is what makes "the structural-validation tests pass"
mean something more than "passed on whatever upstream commit happened to
be current the day this notebook was built."

Pure data plus one formatting helper. No I/O, no imports beyond stdlib.
"""
from __future__ import annotations

REPO_URL = "https://github.com/comfyanonymous/ComfyUI"
TESTED_REF = "806e092ed42772e4ce7abf44c97c50021cc4bd10"
TESTED_DATE = "2026-07-26"


def short_ref(ref: str = TESTED_REF, length: int = 12) -> str:
    """First `length` characters of a commit SHA, for compact printing."""
    return ref[:length]

In [ ]:
%%writefile colab_studio/registry.py
"""Model registry. Pure data -- no network, no filesystem.

Every entry below was HEAD-verified against huggingface.co on 2026-07-25.
Sizes are real, not estimates.
"""
from __future__ import annotations

import os
from dataclasses import dataclass


@dataclass(frozen=True)
class ModelSpec:
    repo: str
    filename: str
    dest_subdir: str          # relative to ComfyUI models/
    size_gb: float
    dest_filename: str | None = None

    @property
    def target_name(self) -> str:
        """Filename as written to disk. Diffusers-layout repos all use the
        same generic filename, so those entries must rename."""
        return self.dest_filename or os.path.basename(self.filename)


UPSCALER = ModelSpec(
    repo="Kim2091/UltraSharp",
    filename="4x-UltraSharp.pth",
    dest_subdir="upscale_models",
    size_gb=0.06,
)

CONTROLNET_CANNY = ModelSpec(
    repo="diffusers/controlnet-canny-sdxl-1.0",
    filename="diffusion_pytorch_model.fp16.safetensors",
    dest_subdir="controlnet",
    size_gb=2.33,
    dest_filename="controlnet-canny-sdxl.safetensors",
)

PROFILES: dict[str, list[ModelSpec]] = {
    # No standalone VAE: every builder in workflows.py takes its VAE from
    # CheckpointLoaderSimple output 2, and none of them emits a VAELoader, so
    # a separate sdxl-vae download is 0.31 GB that nothing ever opens.
    "sdxl": [
        ModelSpec(
            repo="stabilityai/stable-diffusion-xl-base-1.0",
            filename="sd_xl_base_1.0.safetensors",
            dest_subdir="checkpoints",
            size_gb=6.46,
        ),
    ],
    # fp8 all-in-one: UNet + T5 + CLIP-L + VAE in one file, so it loads
    # via CheckpointLoaderSimple rather than a 3-loader split.
    "flux-dev": [
        ModelSpec(
            repo="Comfy-Org/flux1-dev",
            filename="flux1-dev-fp8.safetensors",
            dest_subdir="checkpoints",
            size_gb=16.06,
        ),
    ],
    "flux-schnell": [
        ModelSpec(
            repo="Comfy-Org/flux1-schnell",
            filename="flux1-schnell-fp8.safetensors",
            dest_subdir="checkpoints",
            size_gb=16.05,
        ),
    ],
}

CHECKPOINT_NAME: dict[str, str] = {
    "sdxl": "sd_xl_base_1.0.safetensors",
    "flux-dev": "flux1-dev-fp8.safetensors",
    "flux-schnell": "flux1-schnell-fp8.safetensors",
}


def resolve(profile: str, controlnet: bool = False,
            upscale: bool = True) -> list[ModelSpec]:
    """Full download list for a profile.

    Raises KeyError on an unknown profile, and ValueError for
    flux + controlnet: CONTROLNET_CANNY holds SDXL weights that cannot
    condition a Flux checkpoint, so returning them would mean downloading
    2.33 GB that workflows.controlnet_canny() will then refuse to use.
    """
    specs = list(PROFILES[profile])
    if controlnet and profile.startswith("flux"):
        raise ValueError(
            f"ControlNet is SDXL-only: {CONTROLNET_CANNY.repo} ships SDXL "
            f"weights, unusable with the {profile!r} checkpoint. Flux "
            "ControlNet (flux1-canny-dev, 22.17 GB) is out of scope -- use "
            "profile='sdxl' or controlnet=False."
        )
    if upscale:
        specs.append(UPSCALER)
    if controlnet:
        specs.append(CONTROLNET_CANNY)
    return specs


def total_gb(specs: list[ModelSpec]) -> float:
    return round(sum(s.size_gb for s in specs), 2)

In [ ]:
%%writefile colab_studio/advice.py
"""Map detected GPU VRAM to a model profile, resolution ceiling and
ComfyUI launch flags.

Colab reassigns GPU tiers without warning, so nothing here may be
hardcoded to a subscription level. Thresholds are calibration targets
(spec E4) -- adjust the constants, not the structure.
"""
from __future__ import annotations

from dataclasses import dataclass, field

FLUX_DISK_GB = 16.06     # flux1-dev-fp8.safetensors, HEAD-verified
DISK_HEADROOM_GB = 8.0   # room for outputs, pip wheels, HF temp files

LOW_MAX_VRAM = 12.0
MID_MAX_VRAM = 20.0


@dataclass(frozen=True)
class Advice:
    tier: str
    profile: str
    max_side: int
    launch_flags: list[str] = field(default_factory=list)
    notes: list[str] = field(default_factory=list)


def recommend(vram_gb: float, disk_free_gb: float = 100.0) -> Advice:
    notes: list[str] = []

    if vram_gb < LOW_MAX_VRAM:
        tier, profile, max_side = "low", "sdxl", 768
        flags = ["--normalvram"]
        notes.append(
            f"{vram_gb:.1f} GB VRAM is tight for SDXL. Capped at 768px; "
            "expect OOM at 1024 with a large batch."
        )
    elif vram_gb < MID_MAX_VRAM:
        tier, profile, max_side = "mid", "sdxl", 1024
        flags = ["--normalvram"]
        notes.append(
            "SDXL at 1024px is comfortable. Flux fp8 (16 GB) will not fit "
            "alongside activations -- not offered at this tier."
        )
    else:
        tier, profile, max_side = "high", "flux-dev", 1024
        flags = ["--highvram"]
        notes.append("Enough VRAM for Flux dev fp8 and SDXL at 1024px.")

    # Disk guard: if model won't fit, downgrade profile but keep tier/flags.
    # Tier and launch_flags reflect available VRAM; profile reflects what will fit on disk.
    if profile.startswith("flux") and disk_free_gb < FLUX_DISK_GB + DISK_HEADROOM_GB:
        notes.append(
            f"Only {disk_free_gb:.0f} GB disk free; Flux needs "
            f"{FLUX_DISK_GB:.0f} GB plus headroom. Falling back to SDXL."
        )
        profile = "sdxl"

    return Advice(tier=tier, profile=profile, max_side=max_side,
                  launch_flags=flags, notes=notes)

In [ ]:
%%writefile colab_studio/workflows.py
"""API-format graph builders for ComfyUI.

API format is a flat dict: {node_id: {"class_type": str, "inputs": {...}}}.
Links are ["source_node_id", output_index] pairs.

Input key names come from INPUT_TYPES() on ComfyUI 0.10.0. They are not
guessable -- change them only against a fresh dump.
"""
from __future__ import annotations

Graph = dict


def _base(ckpt: str, prompt: str, negative: str, seed: int, steps: int,
          cfg: float, sampler: str, scheduler: str, denoise: float,
          prefix: str) -> Graph:
    """Shared spine: checkpoint -> two text encodes -> sampler -> decode -> save.

    Node "4" (the latent source) is deliberately left out; each builder
    supplies either EmptyLatentImage or VAEEncode.
    """
    return {
        "1": {"class_type": "CheckpointLoaderSimple",
              "inputs": {"ckpt_name": ckpt}},
        "2": {"class_type": "CLIPTextEncode",
              "inputs": {"text": prompt, "clip": ["1", 1]}},
        "3": {"class_type": "CLIPTextEncode",
              "inputs": {"text": negative, "clip": ["1", 1]}},
        "5": {"class_type": "KSampler",
              "inputs": {"model": ["1", 0], "seed": seed, "steps": steps,
                         "cfg": cfg, "sampler_name": sampler,
                         "scheduler": scheduler, "positive": ["2", 0],
                         "negative": ["3", 0], "latent_image": ["4", 0],
                         "denoise": denoise}},
        "6": {"class_type": "VAEDecode",
              "inputs": {"samples": ["5", 0], "vae": ["1", 2]}},
        "7": {"class_type": "SaveImage",
              "inputs": {"images": ["6", 0], "filename_prefix": prefix}},
    }


def _empty_latent(width: int, height: int, batch: int) -> Graph:
    return {"class_type": "EmptyLatentImage",
            "inputs": {"width": width, "height": height, "batch_size": batch}}


def is_flux(profile: str) -> bool:
    """True for every Flux profile name in registry.PROFILES."""
    return profile.startswith("flux")


def _spine(ckpt: str, prompt: str, negative: str, seed: int, steps: int,
           cfg: float, sampler: str, scheduler: str, denoise: float,
           prefix: str, profile: str, guidance: float) -> Graph:
    """`_base` plus the profile-dependent sampling contract.

    THE single place the Flux-vs-SDXL decision lives. Flux ignores classifier
    free guidance entirely: cfg must be 1.0 with the real guidance supplied by
    a FluxGuidance node wired into KSampler.positive. Any other cfg produces
    scorched output, so cfg/sampler/scheduler are overridden rather than
    trusted -- every builder routes through here so no graph can be handed a
    Flux checkpoint with SDXL-shaped sampling.
    """
    if is_flux(profile):
        cfg, sampler, scheduler = 1.0, "euler", "simple"
    g = _base(ckpt, prompt, negative, seed, steps, cfg, sampler, scheduler,
              denoise, prefix)
    if is_flux(profile):
        g["8"] = {"class_type": "FluxGuidance",
                  "inputs": {"conditioning": ["2", 0], "guidance": guidance}}
        g["5"]["inputs"]["positive"] = ["8", 0]
    return g


def _txt2img(ckpt: str, prompt: str, negative: str, seed: int, steps: int,
             cfg: float, width: int, height: int, batch: int, sampler: str,
             scheduler: str, prefix: str, profile: str,
             guidance: float) -> Graph:
    g = _spine(ckpt, prompt, negative, seed, steps, cfg, sampler, scheduler,
               1.0, prefix, profile, guidance)
    g["4"] = _empty_latent(width, height, batch)
    return g


def sdxl_txt2img(ckpt: str, prompt: str, negative: str = "", seed: int = 0,
                 steps: int = 25, cfg: float = 7.0, width: int = 1024,
                 height: int = 1024, batch: int = 1,
                 sampler: str = "dpmpp_2m", scheduler: str = "karras") -> Graph:
    """SDXL txt2img: 7 nodes, dpmpp_2m/karras. Use flux_txt2img for Flux."""
    return _txt2img(ckpt, prompt, negative, seed, steps, cfg, width, height,
                    batch, sampler, scheduler, "colab/sdxl", "sdxl", 3.5)


def flux_txt2img(ckpt: str, prompt: str, negative: str = "", seed: int = 0,
                 steps: int = 20, cfg: float = 1.0, width: int = 1024,
                 height: int = 1024, batch: int = 1,
                 guidance: float = 3.5) -> Graph:
    """Flux ignores CFG entirely -- it must be 1.0, with real guidance
    supplied by FluxGuidance. Any other cfg produces scorched output, so the
    parameter is overridden rather than trusted (see _spine)."""
    return _txt2img(ckpt, prompt, negative, seed, steps, cfg, width, height,
                    batch, "euler", "simple", "colab/flux", "flux", guidance)


def img2img(ckpt: str, prompt: str, image: str, negative: str = "",
            seed: int = 0, steps: int = 25, cfg: float = 7.0,
            denoise: float = 0.6, sampler: str = "dpmpp_2m",
            scheduler: str = "karras", profile: str = "sdxl",
            guidance: float = 3.5) -> Graph:
    """`image` is a filename already present in the server's input/ dir --
    upload it first via ComfyClient.upload_image().

    Pass `profile` so a Flux checkpoint gets Flux sampling; the default is
    SDXL-shaped.
    """
    g = _spine(ckpt, prompt, negative, seed, steps, cfg, sampler, scheduler,
               denoise, "colab/img2img", profile, guidance)
    g["10"] = {"class_type": "LoadImage", "inputs": {"image": image}}
    g["4"] = {"class_type": "VAEEncode",
              "inputs": {"pixels": ["10", 0], "vae": ["1", 2]}}
    return g


def upscale(ckpt: str, prompt: str, negative: str = "", seed: int = 0,
            steps: int = 25, cfg: float = 7.0, width: int = 1024,
            height: int = 1024, batch: int = 1,
            model_name: str = "4x-UltraSharp.pth",
            sampler: str = "dpmpp_2m", scheduler: str = "karras",
            profile: str = "sdxl", guidance: float = 3.5) -> Graph:
    """txt2img then a pure image-space upscale. No image input, so this is
    the one optional feature needing no upload path.

    Pass `profile` so a Flux checkpoint gets Flux sampling; the default is
    SDXL-shaped.
    """
    g = _txt2img(ckpt, prompt, negative, seed, steps, cfg, width, height,
                 batch, sampler, scheduler, "colab/upscale", profile, guidance)
    g["11"] = {"class_type": "UpscaleModelLoader",
               "inputs": {"model_name": model_name}}
    g["12"] = {"class_type": "ImageUpscaleWithModel",
               "inputs": {"upscale_model": ["11", 0], "image": ["6", 0]}}
    g["7"]["inputs"]["images"] = ["12", 0]
    return g


def controlnet_canny(ckpt: str, prompt: str, image: str, negative: str = "",
                     seed: int = 0, steps: int = 25, cfg: float = 7.0,
                     width: int = 1024, height: int = 1024, batch: int = 1,
                     strength: float = 0.8, low_threshold: float = 0.4,
                     high_threshold: float = 0.8,
                     control_net: str = "controlnet-canny-sdxl.safetensors",
                     sampler: str = "dpmpp_2m",
                     scheduler: str = "karras",
                     profile: str = "sdxl") -> Graph:
    """SDXL only. Canny is a core node -- no comfyui_controlnet_aux needed.

    ControlNetApplyAdvanced emits BOTH conditionings, so the sampler's
    positive and negative must be rewired to outputs 0 and 1 of the same
    node. Rewiring only positive is a silent correctness bug.

    Raises ValueError for a Flux profile: the canny weights really are SDXL
    (registry.CONTROLNET_CANNY), and Flux ControlNet is out of scope at
    22.17 GB. There is no Flux-correct graph to fall back to, so this refuses
    rather than silently producing a wrong one.
    """
    if is_flux(profile):
        raise ValueError(
            f"controlnet_canny is SDXL-only: {control_net!r} holds SDXL "
            f"weights and cannot condition the {profile!r} checkpoint "
            f"{ckpt!r}. Flux ControlNet (flux1-canny-dev, 22.17 GB) is out of "
            "scope -- use profile='sdxl', or img2img() for a Flux edit."
        )
    g = _txt2img(ckpt, prompt, negative, seed, steps, cfg, width, height,
                 batch, sampler, scheduler, "colab/controlnet", profile, 3.5)
    g["10"] = {"class_type": "LoadImage", "inputs": {"image": image}}
    g["13"] = {"class_type": "Canny",
               "inputs": {"image": ["10", 0], "low_threshold": low_threshold,
                          "high_threshold": high_threshold}}
    g["14"] = {"class_type": "ControlNetLoader",
               "inputs": {"control_net_name": control_net}}
    g["15"] = {"class_type": "ControlNetApplyAdvanced",
               "inputs": {"positive": ["2", 0], "negative": ["3", 0],
                          "control_net": ["14", 0], "image": ["13", 0],
                          "strength": strength, "start_percent": 0.0,
                          "end_percent": 1.0}}
    g["5"]["inputs"]["positive"] = ["15", 0]
    g["5"]["inputs"]["negative"] = ["15", 1]
    return g

In [ ]:
%%writefile colab_studio/fetch.py
"""Download ModelSpecs into a ComfyUI models/ tree.

Uses local_dir= so huggingface_hub writes straight to the destination.
The previous implementation downloaded to the HF cache and then copied,
doubling disk use -- fatal for a 16 GB checkpoint on a Colab VM.
"""
from __future__ import annotations

import os
import shutil  # noqa: F401 -- referenced only via monkeypatch in fetch_test.py
from typing import Callable

from huggingface_hub import hf_hub_download

from colab_studio.registry import ModelSpec

Emit = Callable[[str], None]


def _noop(_: str) -> None:
    return None


def download(spec: ModelSpec, models_dir: str, emit: Emit | None = None) -> str:
    """Fetch one spec. Returns the final on-disk path. Idempotent."""
    log = emit or _noop
    dest_dir = os.path.join(models_dir, spec.dest_subdir)
    os.makedirs(dest_dir, exist_ok=True)
    final = os.path.join(dest_dir, spec.target_name)

    if os.path.exists(final):
        log(f"[=] {spec.target_name} already present, skipping")
        return final

    log(f"[+] {spec.target_name} ({spec.size_gb:.2f} GB) from {spec.repo}")
    got = hf_hub_download(
        repo_id=spec.repo,
        filename=spec.filename,
        local_dir=dest_dir,
    )

    # Nested filenames land in a subtree, and diffusers-layout repos all use
    # the same generic name; flatten and rename to the target.
    if os.path.abspath(got) != os.path.abspath(final):
        os.replace(got, final)
        stray = os.path.dirname(got)
        while os.path.abspath(stray) != os.path.abspath(dest_dir):
            try:
                os.rmdir(stray)
            except OSError:
                break
            stray = os.path.dirname(stray)

    log(f"[v] {spec.target_name} ready")
    return final


def download_all(specs: list[ModelSpec], models_dir: str,
                 emit: Emit | None = None) -> list[str]:
    return [download(s, models_dir, emit) for s in specs]

In [ ]:
%%writefile colab_studio/client.py
"""HTTP client for a running ComfyUI server.

Deliberately talks HTTP rather than importing comfy: comfy/cli_args.py:236
parses sys.argv when args_parsing is enabled, and comfy/model_management.py:238
probes the GPU at import time. Both are hostile inside a notebook kernel.
"""
from __future__ import annotations

import os
import time
import uuid
from typing import Callable

import requests


class ComfyError(RuntimeError):
    """Server rejected a request. Carries node_errors when present."""


def _execution_error(hist: dict) -> str | None:
    """Failure detail from a /history entry, or None if it has not failed.

    Only `status_str == "error"` counts. `completed` is legitimately False
    while a job is still running, so keying on it would abort every generate
    on the first poll.
    """
    status = hist.get("status") or {}
    if status.get("status_str") != "error":
        return None
    for message in status.get("messages") or []:
        if not (isinstance(message, (list, tuple)) and len(message) == 2):
            continue
        name, payload = message
        if name not in ("execution_error", "execution_interrupted"):
            continue
        if not isinstance(payload, dict):
            continue
        return (
            f"{payload.get('exception_type') or name}: "
            f"{payload.get('exception_message') or '(no message)'} "
            f"[node {payload.get('node_id')} "
            f"{payload.get('node_type')}]"
        )
    return "server reported status_str=error with no execution_error message"


class ComfyClient:
    def __init__(self, base_url: str = "http://127.0.0.1:8188") -> None:
        self.base_url = base_url.rstrip("/")
        self.client_id = str(uuid.uuid4())

    def wait_ready(self, timeout: float = 180.0, interval: float = 1.0) -> bool:
        """Poll /system_stats until the server answers. Start the tunnel only
        after this returns True, or the public URL 502s."""
        deadline = time.time() + timeout
        while time.time() < deadline:
            try:
                r = requests.get(f"{self.base_url}/system_stats", timeout=5)
                if r.status_code == 200:
                    return True
            except (requests.exceptions.MissingSchema,
                    requests.exceptions.InvalidSchema,
                    requests.exceptions.InvalidURL) as err:
                raise ComfyError(f"invalid base_url {self.base_url!r}: {err}") from err
            except requests.RequestException:
                pass
            time.sleep(interval)
        return False

    def system_stats(self) -> dict:
        """GET /system_stats -- the server's own device inventory and memory
        counters (server.py's system_stats handler; devices[0] is the
        primary device). This is the only window into VRAM used by the
        diffusion model, which runs in this server's process, not the
        notebook kernel -- see colab_studio.telemetry for how it is turned
        into an (honestly qualified) observed-peak reading."""
        r = requests.get(f"{self.base_url}/system_stats", timeout=10)
        if r.status_code != 200:
            raise ComfyError(f"system_stats failed ({r.status_code}): {r.text[:300]}")
        return r.json()

    def upload_image(self, path: str) -> str:
        """Upload to the server's input/ dir. Returns the name to put in
        LoadImage.inputs.image."""
        with open(path, "rb") as fh:
            r = requests.post(
                f"{self.base_url}/upload/image",
                files={"image": (os.path.basename(path), fh)},
                data={"overwrite": "true"},
                timeout=120,
            )
        if r.status_code != 200:
            raise ComfyError(f"upload failed ({r.status_code}): {r.text[:300]}")
        return r.json()["name"]

    def submit(self, graph: dict) -> str:
        r = requests.post(
            f"{self.base_url}/prompt",
            json={"prompt": graph, "client_id": self.client_id},
            timeout=60,
        )
        if r.status_code != 200:
            try:
                payload = r.json()
            except ValueError:
                raise ComfyError(f"submit failed ({r.status_code}): {r.text[:300]}")
            raise ComfyError(
                f"submit rejected: {payload.get('error')} "
                f"node_errors={payload.get('node_errors')}"
            )
        return r.json()["prompt_id"]

    def wait_result(self, prompt_id: str, timeout: float = 600.0,
                    interval: float = 1.0,
                    on_poll: Callable[[], None] | None = None) -> list[dict]:
        """Poll /history until outputs appear. Returns image refs.

        Raises ComfyError as soon as the server reports an execution failure.
        Structural rejections come back from submit()'s 400 path, but runtime
        failures (OOM, tensor mismatch) only ever show up here -- and they
        leave `outputs` empty forever, so ignoring `status` means a silent
        full-timeout hang ending in a diagnostic-free TimeoutError.

        `on_poll`, when given, is called once per pass through this loop --
        the natural point to sample telemetry (see
        colab_studio.telemetry.VramProbe) at the same cadence as `interval`.
        Its exceptions are swallowed: a telemetry failure must never be able
        to abort a render that may otherwise run for many minutes.
        """
        deadline = time.time() + timeout
        while time.time() < deadline:
            if on_poll is not None:
                try:
                    on_poll()
                except Exception:
                    pass
            r = requests.get(f"{self.base_url}/history/{prompt_id}", timeout=15)
            if r.status_code == 200:
                hist = r.json().get(prompt_id)
                if hist:
                    detail = _execution_error(hist)
                    if detail:
                        raise ComfyError(
                            f"execution failed for {prompt_id}: {detail}")
                    if hist.get("outputs"):
                        refs: list[dict] = []
                        for node_out in hist["outputs"].values():
                            refs.extend(node_out.get("images", []))
                        if refs:
                            return refs
            time.sleep(interval)
        raise TimeoutError(f"no outputs for {prompt_id} within {timeout}s")

    def fetch_image(self, ref: dict) -> bytes:
        r = requests.get(
            f"{self.base_url}/view",
            params={"filename": ref["filename"],
                    "subfolder": ref.get("subfolder", ""),
                    "type": ref.get("type", "output")},
            timeout=120,
        )
        if r.status_code != 200:
            raise ComfyError(f"view failed ({r.status_code})")
        return r.content

    def generate(self, graph: dict, timeout: float = 600.0,
                on_poll: Callable[[], None] | None = None) -> list[bytes]:
        """submit -> wait -> fetch. The whole inline-cell path in one call.

        `on_poll` is threaded straight through to wait_result() -- pass
        VramProbe.sample for VRAM telemetry sampled once per poll.
        """
        pid = self.submit(graph)
        return [self.fetch_image(ref)
                for ref in self.wait_result(pid, timeout, on_poll=on_poll)]

In [ ]:
%%writefile colab_studio/telemetry.py
"""Observed-peak VRAM sampling for a running ComfyUI server subprocess.

The diffusion model runs in ComfyUI's server subprocess, not the notebook
kernel: `torch.cuda.max_memory_allocated()` called from a notebook cell
would see only the kernel's own (empty) CUDA allocator and report ~0. The
only window into the server's memory is the server's own GET
/system_stats endpoint (server.py; `devices[0]` is the primary device;
used bytes = `vram_total - vram_free`).

VramProbe is meant to be driven from ComfyClient.wait_result's `on_poll`
hook -- i.e. sampled once per poll interval (1s by default). That makes
every number this module produces an OBSERVED PEAK AT N-SECOND SAMPLING,
NOT A TRUE MAXIMUM: a spike between two polls is invisible to it, and
model load can peak higher than steady-state sampling ever catches. Never
present or label this number as an exact ceiling -- `describe()` below
always states the sampling interval and the word "observed" for exactly
this reason, and any other surface printing this value must do the same.
"""
from __future__ import annotations

from dataclasses import dataclass

from colab_studio.client import ComfyClient

_BYTES_PER_GB = 2**30


@dataclass(frozen=True)
class VramSummary:
    """A snapshot of what VramProbe has observed so far. See the module
    docstring: `peak_used_gb` is a polled sample, not a true maximum."""
    device_name: str | None
    peak_used_gb: float
    total_gb: float
    percent_used: float
    sample_count: int


class VramProbe:
    """Polled VRAM sampler, driven externally (e.g. by `on_poll`).

    Degrades silently in every case that isn't a genuine reading: no
    devices reported, a non-GPU device (`type == "cpu"`), a response
    missing the expected keys, or a request that fails outright all just
    mean "no sample was recorded" -- none of them raise. A telemetry
    failure must never be able to interrupt a render.

    All I/O goes through the injected ComfyClient, so this is testable
    against a plain stub HTTP server with no GPU involved.
    """

    def __init__(self, client: ComfyClient) -> None:
        self._client = client
        self._device_name: str | None = None
        self._total_bytes = 0
        self._peak_used_bytes = 0
        self._sample_count = 0

    def sample(self) -> None:
        """Record one observation. Never raises."""
        try:
            stats = self._client.system_stats()
            devices = stats.get("devices") or []
            if not devices:
                return
            device = devices[0]
            if not isinstance(device, dict) or device.get("type") == "cpu":
                return
            total = device["vram_total"]
            free = device["vram_free"]
            used = total - free
            name = device.get("name")
        except Exception:
            return

        self._device_name = name or self._device_name
        self._total_bytes = total
        self._peak_used_bytes = max(self._peak_used_bytes, used)
        self._sample_count += 1

    def summary(self) -> VramSummary:
        """The peak observed so far. `sample_count == 0` means no usable
        reading was ever taken -- report that honestly, don't fake a zero."""
        if self._sample_count == 0:
            return VramSummary(device_name=None, peak_used_gb=0.0,
                               total_gb=0.0, percent_used=0.0, sample_count=0)
        percent = (self._peak_used_bytes / self._total_bytes * 100.0
                  if self._total_bytes else 0.0)
        return VramSummary(
            device_name=self._device_name,
            peak_used_gb=self._peak_used_bytes / _BYTES_PER_GB,
            total_gb=self._total_bytes / _BYTES_PER_GB,
            percent_used=percent,
            sample_count=self._sample_count,
        )


def describe(summary: VramSummary, interval_s: float = 1.0) -> str:
    """One readable line for a summary. Always names this an *observed*
    peak and states the sampling interval -- see the module docstring for
    why that qualifier may never be dropped."""
    if summary.sample_count == 0:
        return ("no VRAM samples were taken (no GPU reported by the server, "
                "or /system_stats was unavailable) -- peak VRAM is "
                "unmeasured for this run.")
    return (
        f"observed peak VRAM (sampled every {interval_s:g}s): "
        f"{summary.peak_used_gb:.1f} / {summary.total_gb:.1f} GB "
        f"({summary.percent_used:.0f}%) on {summary.device_name}, "
        f"{summary.sample_count} samples"
    )

In [ ]:
%%writefile colab_studio/launch.py
"""Background the ComfyUI server so the notebook kernel stays usable.

The original notebook ended with `!python3 main.py`, which never returns --
every cell after it was unreachable. Backgrounding is what makes the
generate/log/ops cells exist at all.

Ordering matters: server -> wait_ready() -> tunnel. Starting the tunnel
first prints a URL that 502s until the server finishes booting.

RuntimeSupervisor owns that lifecycle end to end (start/wait/stop/restart
the server, start/stop the tunnel, close() both) so it doesn't have to live
in the notebook. The module-level functions below (start_server,
start_tunnel, stop_tunnel, current_tunnel, tail) are thin wrappers over a
shared, module-level default RuntimeSupervisor instance, kept for backward
compatibility with the generated notebook -- their signatures and behaviour
are unchanged.
"""
from __future__ import annotations

import atexit
import os
import re
import subprocess
import sys
import time

from colab_studio.client import ComfyClient

CLOUDFLARED = "/usr/local/bin/cloudflared"
TUNNEL_RE = re.compile(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com")

# Handle on the tunnel started by the last successful start_tunnel().
# start_tunnel() returns only a URL, so without this the notebook holds no
# reference: re-tunnelling would delete the log the running cloudflared still
# has an fd on, spawn a second process on the same port, and leave the first
# alive and unkillable from the kernel.
#
# This is module state rather than per-supervisor state deliberately: a
# cloudflared tunnel is a singleton concept for one kernel (one log file,
# one port scraped via start_new_session=True), so every RuntimeSupervisor
# shares it rather than each tracking its own. Kept as a module global,
# rather than folded into RuntimeSupervisor, so this file's existing tests
# -- which poke launch._TUNNEL directly -- keep working unchanged.
_TUNNEL: subprocess.Popen | None = None


class RuntimeSupervisor:
    """Owns the lifecycle of one ComfyUI server process (plus the shared
    cloudflared tunnel, see _TUNNEL above) for this notebook kernel.

    start_server() refuses to launch a second server while one this
    instance started is still alive: it returns the existing Popen
    unchanged rather than raising. That matches the common case -- a
    notebook cell re-run to "make sure the server is up" -- where the
    caller wants a live process back, not an exception. Call stop_server()
    or restart_server() first to force a fresh one.
    """

    def __init__(self) -> None:
        self._server: subprocess.Popen | None = None
        self._port: int = 8188
        # So a dying kernel doesn't orphan the server or cloudflared: this
        # runs at interpreter exit even if the notebook never calls close().
        atexit.register(self.close)

    # ---------------------------------------------------------- server --

    def start_server(self, comfy_dir: str, flags: list[str], log_path: str,
                     port: int = 8188, python: str | None = None) -> subprocess.Popen:
        """Launch main.py detached, stdout+stderr to log_path. Returns at once.

        Returns the already-running process unchanged if one started by
        this supervisor is still alive -- see class docstring.
        """
        if self._server is not None and self._server.poll() is None:
            return self._server
        exe = python or sys.executable
        cmd = [exe, "main.py", "--listen", "127.0.0.1", "--port", str(port), *flags]
        # The parent's copy of the log fd must be closed once Popen has
        # duplicated it for the child, on every path -- including if Popen
        # itself raises. A `with` block does that regardless of outcome;
        # the fd survives in the child, which has its own duplicate.
        with open(log_path, "wb") as log:
            proc = subprocess.Popen(
                cmd, cwd=comfy_dir, stdout=log, stderr=subprocess.STDOUT,
                start_new_session=True,
            )
        self._server = proc
        self._port = port
        return proc

    def wait_ready(self, timeout: float = 300, client: ComfyClient | None = None) -> bool:
        """Poll until the server answers. Delegates to a ComfyClient aimed
        at this supervisor's port by default; pass one in to override."""
        if client is None:
            client = ComfyClient(f"http://127.0.0.1:{self._port}")
        return client.wait_ready(timeout=timeout)

    def stop_server(self, timeout: float = 30) -> bool:
        """Terminate the server this supervisor started.

        Idempotent: returns False without raising if nothing was ever
        started, or if it had already exited.
        """
        proc, self._server = self._server, None
        if proc is None or proc.poll() is not None:
            return False
        proc.terminate()
        try:
            proc.wait(timeout=timeout)
        except subprocess.TimeoutExpired:
            proc.kill()
            proc.wait(timeout=timeout)
        return True

    def restart_server(self, comfy_dir: str, flags: list[str], log_path: str,
                       port: int = 8188, python: str | None = None,
                       stop_timeout: float = 30) -> subprocess.Popen:
        """stop_server() the current process, then start_server() a new one."""
        self.stop_server(timeout=stop_timeout)
        return self.start_server(comfy_dir, flags, log_path, port=port, python=python)

    # ---------------------------------------------------------- tunnel --

    def current_tunnel(self) -> subprocess.Popen | None:
        """The cloudflared process from the last start_tunnel(), if still alive."""
        if _TUNNEL is not None and _TUNNEL.poll() is None:
            return _TUNNEL
        return None

    def stop_tunnel(self, timeout: float = 10.0) -> bool:
        """Terminate the tunnel from the last start_tunnel().

        Idempotent: returns False without raising if no tunnel is running.
        """
        global _TUNNEL
        proc, _TUNNEL = _TUNNEL, None
        if proc is None or proc.poll() is not None:
            return False
        proc.terminate()
        try:
            proc.wait(timeout=timeout)
        except subprocess.TimeoutExpired:
            proc.kill()
        return True

    def start_tunnel(self, port: int, log_path: str, timeout: float = 40.0,
                     interval: float = 1.0) -> str | None:
        """Start cloudflared and scrape the public URL out of its log.

        Call only after wait_ready() returns True.

        Any tunnel from a previous call is stopped first: the log is
        truncated here, so leaving the old process holding an fd on it
        would orphan a second cloudflared on the same port with no way to
        reach it from the notebook.
        """
        global _TUNNEL
        self.stop_tunnel()
        if os.path.exists(log_path):
            os.remove(log_path)
        try:
            # Closed via `with` in every path, including if Popen raises --
            # the old code left this fd open on the FileNotFoundError path.
            with open(log_path, "wb") as log:
                proc = subprocess.Popen(
                    [CLOUDFLARED, "tunnel", "--url", f"http://127.0.0.1:{port}"],
                    stdout=log, stderr=subprocess.STDOUT, start_new_session=True,
                )
        except FileNotFoundError:
            return None
        # Recorded before the scrape loop so a caller can always stop what
        # was started, including on the timeout path below.
        _TUNNEL = proc
        deadline = time.time() + timeout
        while time.time() < deadline:
            if proc.poll() is not None:
                _TUNNEL = None
                return None
            try:
                with open(log_path, "r", errors="ignore") as fh:
                    m = TUNNEL_RE.search(fh.read())
                if m:
                    # start_new_session=True: the tunnel must outlive this cell.
                    return m.group(0)
            except FileNotFoundError:
                pass
            time.sleep(interval)
        proc.terminate()
        _TUNNEL = None
        return None

    # ------------------------------------------------------------ close --

    def close(self) -> None:
        """Stop both the server and the tunnel. Idempotent -- safe to call
        more than once, and registered via atexit so a dying kernel doesn't
        orphan either process."""
        self.stop_server()
        self.stop_tunnel()


# Backs the module-level functions below. Shared across the process so the
# generated notebook, which only ever calls the free functions, gets the
# hardened lifecycle (single-server guard, closed log handles, atexit
# cleanup) without having to know RuntimeSupervisor exists.
_SUPERVISOR = RuntimeSupervisor()


def start_server(comfy_dir: str, flags: list[str], log_path: str,
                 port: int = 8188, python: str | None = None) -> subprocess.Popen:
    """Launch main.py detached, stdout+stderr to log_path. Returns at once."""
    return _SUPERVISOR.start_server(comfy_dir, flags, log_path, port=port, python=python)


def current_tunnel() -> subprocess.Popen | None:
    """The cloudflared process from the last start_tunnel(), if still alive."""
    return _SUPERVISOR.current_tunnel()


def stop_tunnel(timeout: float = 10.0) -> bool:
    """Terminate the tunnel from the last start_tunnel().

    Returns True if a live process was stopped. Safe to call when no tunnel
    is running.
    """
    return _SUPERVISOR.stop_tunnel(timeout=timeout)


def start_tunnel(port: int, log_path: str, timeout: float = 40.0,
                 interval: float = 1.0) -> str | None:
    """Start cloudflared and scrape the public URL out of its log.

    Call only after ComfyClient.wait_ready() returns True.

    Any tunnel from a previous call is stopped first: the log is truncated
    here, so leaving the old process holding an fd on it would orphan a second
    cloudflared on the same port with no way to reach it from the notebook.
    """
    return _SUPERVISOR.start_tunnel(port, log_path, timeout=timeout, interval=interval)


def tail(log_path: str, n: int = 40) -> str:
    """Last n lines of a logfile. Empty string if it does not exist yet."""
    try:
        with open(log_path, "r", errors="ignore") as fh:
            return "\n".join(fh.read().splitlines()[-n:])
    except FileNotFoundError:
        return ""

In [ ]:
#@title 5. Choose profile and download models
import os, shutil, sys
sys.path.insert(0, COMFY_DIR)
from colab_studio.advice import recommend
from colab_studio.registry import resolve, total_gb, CHECKPOINT_NAME
from colab_studio.fetch import download_all

MODELS_DIR = os.path.join(COMFY_DIR, "models")
os.makedirs(MODELS_DIR, exist_ok=True)
# Measure the filesystem the checkpoints actually land on. With
# PERSIST="everything" cell 4 symlinked models/ to Drive, whose free tier is
# 15 GB -- less than one Flux checkpoint. Cell 2's /content figure would
# happily approve a download that cannot fit.
MODELS_FREE_GB = shutil.disk_usage(MODELS_DIR).free / 2**30
print(f"models/ lands on a filesystem with {MODELS_FREE_GB:.0f} GB free")

ADVICE = recommend(VRAM_GB, MODELS_FREE_GB)
PROFILE = ADVICE.profile if IMAGE_MODEL == "auto" else IMAGE_MODEL
LAUNCH_FLAGS = ADVICE.launch_flags
MAX_SIDE = ADVICE.max_side

if IMAGE_MODEL != "auto" and IMAGE_MODEL != ADVICE.profile:
    print("!" * 70)
    print(f"!! OVERRIDE: you picked '{IMAGE_MODEL}', but this runtime was "
          f"sized for '{ADVICE.profile}'.")
    print("!! Launch flags and the resolution cap still follow the recommended")
    print("!! profile, so expect a failed download or an OOM at model load.")
    for n in ADVICE.notes:
        print("!!  -", n)
    print(f"!! Set IMAGE_MODEL='auto' in cell 1 for '{ADVICE.profile}'.")
    print("!" * 70)

print(f"tier={ADVICE.tier}  profile={PROFILE}  max_side={MAX_SIDE}")
print(f"launch flags: {' '.join(LAUNCH_FLAGS) or '(none)'}")
for n in ADVICE.notes:
    print(" -", n)

# The canny weights are SDXL. On a Flux profile they are 2.33 GB that no
# workflow can use, so drop them rather than download them. Every cell below
# reads USE_CONTROLNET, not the raw form field.
USE_CONTROLNET = CONTROLNET
if USE_CONTROLNET and PROFILE.startswith("flux"):
    print(f"!! ControlNet is SDXL-only; disabled for profile '{PROFILE}'. "
          "Skipping a 2.33 GB unusable download.")
    USE_CONTROLNET = False

SPECS = resolve(PROFILE, controlnet=USE_CONTROLNET, upscale=USE_UPSCALER)
print(f"\nDownloading {len(SPECS)} files, {total_gb(SPECS)} GB total")
download_all(SPECS, MODELS_DIR, emit=print)
CKPT = CHECKPOINT_NAME[PROFILE]

In [ ]:
#@title 6. Write API workflows for the generate cells
# API format only -- that is what cells 8/8b POST to /prompt. These are not
# ComfyUI sidebar workflows: the sidebar wants UI format, which is a different
# schema and is out of scope. In the tunnel UI, build graphs by hand.
import json, os
from colab_studio import workflows

API_DIR = "/content/wf_api"
os.makedirs(API_DIR, exist_ok=True)

# profile=PROFILE is what keeps a Flux checkpoint out of an SDXL-shaped graph.
built = {
    "txt2img": (workflows.flux_txt2img(CKPT, "a prompt")
                if PROFILE.startswith("flux")
                else workflows.sdxl_txt2img(CKPT, "a prompt")),
    "img2img": workflows.img2img(CKPT, "a prompt", image="input.png",
                                 profile=PROFILE),
}
if USE_UPSCALER:
    built["upscale"] = workflows.upscale(CKPT, "a prompt", profile=PROFILE)
if USE_CONTROLNET:
    built["controlnet_canny"] = workflows.controlnet_canny(
        CKPT, "a prompt", image="input.png", profile=PROFILE)

for name, graph in built.items():
    with open(os.path.join(API_DIR, f"{name}.json"), "w") as fh:
        json.dump(graph, fh, indent=1)
print(f"API workflows written to {API_DIR}: " + ", ".join(built))

In [ ]:
#@title 7. Launch server (backgrounded); tunnel only if OPEN_PUBLIC_UI
import os
from colab_studio.client import ComfyClient
from colab_studio.launch import RuntimeSupervisor

# RuntimeSupervisor owns the server + tunnel lifecycle (single-server guard,
# closed log handles, atexit cleanup on kernel death) so this cell doesn't
# have to reimplement any of that. The single-server guard is per-instance
# (see launch.py), so re-running this cell must reuse the same instance --
# a fresh RuntimeSupervisor() on every run would have no memory of the
# server the previous run started, spawn a second one that fails to bind
# the port, and then silently report "ready" off the first, orphaned one.
try:
    SUPERVISOR
except NameError:
    SUPERVISOR = RuntimeSupervisor()

SERVER_LOG = "/content/comfyui.log"
TUNNEL_LOG = "/content/cloudflared.log"

SERVER = SUPERVISOR.start_server(COMFY_DIR, LAUNCH_FLAGS, SERVER_LOG, port=PORT)
CLIENT = ComfyClient(f"http://127.0.0.1:{PORT}")

print("waiting for server...")
if SUPERVISOR.wait_ready(timeout=300, client=CLIENT):
    print("ComfyUI ready.")
    if OPEN_PUBLIC_UI:
        # A Quick Tunnel URL is obscurity, not authentication: anyone who
        # has it (or finds it) can generate, upload, browse output history,
        # and reach ComfyUI's management-adjacent routes. Nothing below
        # secures it -- treat the URL as fully public.
        print("!" * 70)
        print("!! WARNING: OPEN_PUBLIC_UI is on. Starting a public tunnel.")
        print("!! The URL is NOT authentication. Anyone who has it can use")
        print("!! this ComfyUI server: generate, upload, and read history.")
        print("!" * 70)
        if not os.path.isfile("/usr/local/bin/cloudflared"):
            !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
            !chmod +x /usr/local/bin/cloudflared
        # Tunnel only AFTER readiness, or the URL 502s.
        URL = SUPERVISOR.start_tunnel(PORT, TUNNEL_LOG)
        print("Public URL:", URL or "(tunnel failed, see cell 10)")
    else:
        print("OPEN_PUBLIC_UI is off -- no public tunnel started. Use cells "
              "8/8b to generate here, or set OPEN_PUBLIC_UI=True in cell 1 "
              "and rerun this cell for a tunnel.")
else:
    print("server did not come up - run the log cell below")

In [ ]:
#@title 8. Generate an image here (no tunnel needed)
from IPython.display import Image, display
from colab_studio import workflows
from colab_studio.telemetry import VramProbe, describe

prompt = "a lighthouse in a storm, dramatic light"  #@param {type:"string"}
negative = "blurry, watermark, text"  #@param {type:"string"}
steps = 25  #@param {type:"slider", min:4, max:60, step:1}
cfg = 7.0  #@param {type:"number"}
seed = 0  #@param {type:"integer"}
width = 1024  #@param {type:"integer"}
height = 1024  #@param {type:"integer"}
use_upscaler = False  #@param {type:"boolean"}

width, height = min(width, MAX_SIDE), min(height, MAX_SIDE)
kw = dict(prompt=prompt, negative=negative, seed=seed, steps=steps,
          width=width, height=height)
# Profile dispatch lives inside the builders (workflows._spine), so passing
# profile=PROFILE is enough: on a Flux profile the cfg above is overridden to
# 1.0 and a FluxGuidance node is wired in, whichever branch is taken.
if use_upscaler:
    graph = workflows.upscale(CKPT, cfg=cfg, profile=PROFILE, **kw)
elif PROFILE.startswith("flux"):
    graph = workflows.flux_txt2img(CKPT, **kw)
else:
    graph = workflows.sdxl_txt2img(CKPT, cfg=cfg, **kw)
if PROFILE.startswith("flux") and cfg != 1.0:
    print(f"note: Flux ignores cfg; using 1.0 with FluxGuidance, not {cfg}.")

# VRAM telemetry: sampled from GET /system_stats once per wait_result() poll
# (1s by default) -- an OBSERVED PEAK, not a true maximum. See
# colab_studio/telemetry.py for why a spike between polls is invisible to it.
vram_probe = VramProbe(CLIENT)
for i, data in enumerate(CLIENT.generate(graph, on_poll=vram_probe.sample)):
    path = f"/content/gen_{i}.png"
    with open(path, "wb") as fh:
        fh.write(data)
    display(Image(filename=path))
print(describe(vram_probe.summary()))

In [ ]:
#@title 8b. Image-to-image / ControlNet (upload or URL)
run_this = False  #@param {type:"boolean"}
source = "upload"  #@param ["upload", "url"]
image_url = ""  #@param {type:"string"}
mode = "img2img"  #@param ["img2img", "controlnet"]
prompt2 = "an oil painting of the same scene"  #@param {type:"string"}
denoise = 0.6  #@param {type:"slider", min:0.1, max:1.0, step:0.05}

# Gated on purpose: source="upload" opens a file picker that blocks the
# kernel, so an ungated body would stall "Runtime > Run all" here and leave
# every cell below unreachable -- exactly the bug this notebook exists to fix.
if not run_this:
    print("Skipped so Run all can continue. Tick run_this, then run this "
          "cell on its own.")
elif mode == "controlnet" and not USE_CONTROLNET:
    print("ControlNet is not available: tick CONTROLNET in cell 1 (SDXL "
          "profiles only) and rerun cell 5 to fetch the weights.")
else:
    import requests
    from IPython.display import Image, display
    from colab_studio import workflows
    from colab_studio.telemetry import VramProbe, describe

    local = "/content/source_image.png"
    if source == "upload":
        from google.colab import files
        up = files.upload()
        name = next(iter(up))
        with open(local, "wb") as fh:
            fh.write(up[name])
    else:
        with open(local, "wb") as fh:
            fh.write(requests.get(image_url, timeout=60).content)

    server_name = CLIENT.upload_image(local)
    if mode == "controlnet":
        graph = workflows.controlnet_canny(CKPT, prompt2, image=server_name,
                                           profile=PROFILE)
    else:
        graph = workflows.img2img(CKPT, prompt2, image=server_name,
                                  denoise=denoise, profile=PROFILE)

    # See cell 8: VRAM telemetry is an OBSERVED PEAK sampled once per poll,
    # not a true maximum.
    vram_probe = VramProbe(CLIENT)
    for i, data in enumerate(CLIENT.generate(graph, on_poll=vram_probe.sample)):
        path = f"/content/edit_{i}.png"
        with open(path, "wb") as fh:
            fh.write(data)
        display(Image(filename=path))
    print(describe(vram_probe.summary()))

In [ ]:
#@title 9. Server log
from colab_studio.launch import tail
lines = 60  #@param {type:"integer"}
print(tail(SERVER_LOG, n=lines) or "(log empty)")

In [ ]:
#@title 10. Ops - restart, free VRAM, disk, re-tunnel
import os, shutil, requests
action = "disk usage"  #@param ["disk usage", "free VRAM", "restart server", "re-tunnel", "list models"]

if action == "disk usage":
    u = shutil.disk_usage("/content")
    print(f"{u.free/2**30:.1f} GB free of {u.total/2**30:.1f} GB")
elif action == "free VRAM":
    requests.post(f"http://127.0.0.1:{PORT}/free",
                  json={"unload_models": True, "free_memory": True}, timeout=30)
    print("asked ComfyUI to unload models")
elif action == "restart server":
    SERVER = SUPERVISOR.restart_server(COMFY_DIR, LAUNCH_FLAGS, SERVER_LOG, port=PORT)
    print("restarted:", CLIENT.wait_ready(timeout=300))
elif action == "re-tunnel":
    # Same warning as cell 7: the URL is not authentication.
    print("!! Public tunnel: anyone with the URL can use this server.")
    # Kill the old cloudflared first: it holds an fd on TUNNEL_LOG, and two
    # tunnels on one port leaves the first one unreachable from here.
    print("stopped previous tunnel:", SUPERVISOR.stop_tunnel())
    print("URL:", SUPERVISOR.start_tunnel(PORT, TUNNEL_LOG))
else:
    for root, _, fs in os.walk(os.path.join(COMFY_DIR, "models")):
        for f in fs:
            p = os.path.join(root, f)
            print(f"{os.path.getsize(p)/2**30:6.2f} GB  {os.path.relpath(p, COMFY_DIR)}")

## Handbook

### Model sizes and what fits

| Profile | Download | Needs | Notes |
|---|---|---|---|
| `sdxl` | 6.46 GB | ~12 GB VRAM at 1024px | Best all-rounder on a T4 |
| `flux-dev` | 16.1 GB | ~20 GB VRAM | All-in-one fp8: UNet + T5 + CLIP-L + VAE |
| `flux-schnell` | 16.1 GB | ~20 GB VRAM | 4-step; much faster, slightly lower fidelity |
| upscaler | 0.06 GB | negligible | 4x-UltraSharp, image-space |
| ControlNet canny | 2.33 GB | +2 GB VRAM | SDXL only |

### Settings that matter

| Model | steps | cfg | sampler / scheduler |
|---|---|---|---|
| SDXL | 25-30 | 6-8 | `dpmpp_2m` / `karras` |
| Flux dev | 20-25 | **1.0** | `euler` / `simple`, guidance 3.5 |
| Flux schnell | **4** | **1.0** | `euler` / `simple` |

**Flux cfg must be 1.0.** Flux does not use classifier-free guidance; real
guidance rides on the `FluxGuidance` node. Any other cfg scorches the image.

### When it breaks

| Symptom | Cause | Fix |
|---|---|---|
| No public URL printed | `OPEN_PUBLIC_UI` is off by default | Set `OPEN_PUBLIC_UI=True` in cell 1 and rerun cell 7, or run cell 10's `re-tunnel` action. **The URL is not authentication** -- anyone who has it can use this server. |
| Tunnel URL 502s | Server still booting | Rerun cell 7; it waits for readiness first |
| `CUDA out of memory` | Resolution or batch too high | Drop to 768px, batch 1, run "free VRAM" in cell 10 |
| `No such file or directory: ...safetensors` | Download interrupted | Rerun cell 5 - it skips completed files |
| Disk full mid-download | Flux is 16 GB | Use `sdxl`, or set `PERSIST="off"` to reclaim Drive space |
| Generate cell hangs | Server died | Check cell 9 log, then "restart server" in cell 10 |
| `ComfyError: execution failed` | A node raised at runtime | The message names the node and exception; usually OOM - drop resolution |
| Cell 8b did nothing | `run_this` is unticked | Tick it; it defaults off so **Run all** does not stall on the file picker |
| ControlNet disabled on Flux | Canny weights are SDXL | Set `IMAGE_MODEL="sdxl"` in cell 1, or use img2img instead |
| Session dropped | Colab idle timeout | Rerun all; with `PERSIST` on, models and outputs survive |

### Adding a LoRA without leaving Colab

```python
from huggingface_hub import hf_hub_download
hf_hub_download(repo_id="OWNER/REPO", filename="lora.safetensors",
                local_dir=f"{COMFY_DIR}/models/loras")
```

Then use the `LoraLoader` node in the tunnel UI (it is a core node, already
available).